## NLP Assignment 3 — Chatbot using Hugging Face Transformers

## Step 1: Install Required Libraries

In [1]:
# Install required packages
# Run this cell if you're on Google Colab or a fresh environment
!pip install transformers torch --quiet

##  Step 2: Import Libraries

In [2]:
# Import necessary libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

✅ Libraries imported successfully!
PyTorch version: 2.10.0+cu128


## Step 3: Load Pre-trained Model and Tokenizer


In [3]:
# Define the pre-trained model name from Hugging Face Hub
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"⏳ Loading tokenizer from '{MODEL_NAME}'...")
# Load the tokenizer for the model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"⏳ Loading model from '{MODEL_NAME}'... (this may take a moment)")
# Load the pre-trained DialoGPT model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set model to evaluation mode (disables dropout layers — not training)
model.eval()

print("✅ Model and Tokenizer loaded successfully!")

⏳ Loading tokenizer from 'microsoft/DialoGPT-medium'...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model and Tokenizer loaded successfully!


## Step 4: Define the Response Generation Function

In [4]:
def generate_response(user_input, chat_history_ids=None):
    # Encode the user input and append EOS token to mark end of this turn
    # EOS (End-of-Sequence) token acts as a separator between conversation turns
    new_user_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'  # Return as PyTorch tensor
    )

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
    else:
        # First turn: no history yet
        bot_input_ids = new_user_input_ids

    with torch.no_grad():
        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=1000,                          # Max total token length
            pad_token_id=tokenizer.eos_token_id,      # Pad with EOS token
            do_sample=True,                           # Use sampling (not greedy)
            top_p=0.92,                               # Nucleus sampling
            temperature=0.75,                         # Response creativity level
            repetition_penalty=1.3                    # Penalize repeated phrases
        )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True  # Remove special tokens like EOS from output
    )

    return response, chat_history_ids


print("✅ Response generation function defined.")

✅ Response generation function defined.


## Step 5: Run the Chatbot

In [6]:
def run_chatbot():
    """
    Main function to run the interactive console-based chatbot.
    Maintains conversation history across multiple turns.
    Exits when the user types 'exit' or 'quit'.
    """
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("(Type 'exit' or 'quit' to end the conversation)")


    # Initialize conversation history as None (empty at the start)
    chat_history_ids = None

    # Continuous conversation loop
    while True:
        # Prompt the user for input
        user_input = input("\nYou: ").strip()

        # Check for empty input — ask again politely
        if not user_input:
            print("Chatbot: It seems you didn't type anything. Please go ahead!")
            continue

        # Exit condition: user types 'exit' or 'quit' (case-insensitive)
        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: It was great talking to you! Goodbye! 👋")
            print("=" * 60)
            break

        # Generate a response using the model, passing conversation history
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        # Display the chatbot's response
        print(f"\nChatbot: {response}")


# ▶️ Start the chatbot
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?
(Type 'exit' or 'quit' to end the conversation)

You:  What is AI?


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Chatbot: Anime in Japanese. It's a way of writing manga, and it makes the world feel more realistic when you play games like Pokemon or Tales from the Borderlands.

You: Who made C++

Chatbot: Coe has made several games over the years and most recently he is making a mobile game called Cogix that will release sometime this year for iOS Android.

You: exit

Chatbot: It was great talking to you! Goodbye! 👋


In [7]:
import nbformat

nb = nbformat.read("NLP_Assignment_3_Chatbot_Transformers.ipynb", as_version=4)
nb.metadata.pop("widgets", None)
nbformat.write(nb, "FINAL_SUBMISSION.ipynb")

FileNotFoundError: [Errno 2] No such file or directory: 'NLP_Assignment_3_Chatbot_Transformers.ipynb'